# TRICARE PDF(JSON) / CSV → Embedding → Chroma DB (분리 저장 버전)

**한 줄 요약**
- PDF(JSON 기반)는 본문 DB와 표 DB를 분리 저장
- CSV는 coverage / exclusion / other DB를 분리 저장
- 이 노트북은 RAG의 인덱싱 단계(Chroma DB 생성까지)를 수행

# 1. 라이브러리 import

In [1]:
#!pip install langchain-huggingface

In [2]:
import json
import re
from pathlib import Path

import pandas as pd

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 2. 프로젝트 파일 경로 및 DB 이름 설정
- PDF -> JSON 변환한 데이터를 쓴 모델과 쓰지 않은 모델을 비교하기 위해 이름을 다르게 설정

In [3]:
PROJECT_ROOT = Path(r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot")

JSON_DIR = PROJECT_ROOT / "outputs" / "json_docs"
DATA_DIR = PROJECT_ROOT / "data"

# PDF(JSON 기반) DB - 본문 / 표 분리
PDF_TEXT_DB_DIR = PROJECT_ROOT / "tricare_pdf_json_text_db"
PDF_TEXT_COLLECTION_NAME = "tricare_pdf_json_text"

PDF_TABLE_DB_DIR = PROJECT_ROOT / "tricare_pdf_json_table_db"
PDF_TABLE_COLLECTION_NAME = "tricare_pdf_json_table"

# CSV DB - coverage / exclusion / other 분리
CSV_COVERAGE_DB_DIR = PROJECT_ROOT / "tricare_csv_coverage_db"
CSV_COVERAGE_COLLECTION_NAME = "tricare_csv_coverage"

CSV_EXCLUSION_DB_DIR = PROJECT_ROOT / "tricare_csv_exclusion_db"
CSV_EXCLUSION_COLLECTION_NAME = "tricare_csv_exclusion"

CSV_OTHER_DB_DIR = PROJECT_ROOT / "tricare_csv_other_db"
CSV_OTHER_COLLECTION_NAME = "tricare_csv_other"

print("JSON_DIR:", JSON_DIR)
print("DATA_DIR:", DATA_DIR)
print("PDF_TEXT_DB_DIR:", PDF_TEXT_DB_DIR)
print("PDF_TABLE_DB_DIR:", PDF_TABLE_DB_DIR)
print("CSV_COVERAGE_DB_DIR:", CSV_COVERAGE_DB_DIR)
print("CSV_EXCLUSION_DB_DIR:", CSV_EXCLUSION_DB_DIR)
print("CSV_OTHER_DB_DIR:", CSV_OTHER_DB_DIR)

JSON_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs
DATA_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data
PDF_TEXT_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_text_db
PDF_TABLE_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_table_db
CSV_COVERAGE_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_coverage_db
CSV_EXCLUSION_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_exclusion_db
CSV_OTHER_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_other_db


# 3. 공통 전처리 함수
- 공통 전처리 함수

In [4]:
def remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def clean_text(text: str) -> str:
    if not text:
        return ""

    text = str(text)
    text = text.replace("\u00a0", " ")
    text = remove_urls(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def is_noise_text(text: str) -> bool:
    """
    검색 가치가 낮은 블록을 제거하기 위한 함수
    """
    if not text:
        return True

    t = text.strip()

    if len(t) < 30:
        return True

    if t.startswith("http") and " " not in t:
        return True

    digit_ratio = sum(c.isdigit() for c in t) / max(len(t), 1)
    if digit_ratio > 0.35:
        return True

    return False

# 4. 제목 여부를 간단히 판별하는 함수
- 제목 여부를 간단히 판별하는 함수

In [5]:
def is_heading(text: str) -> bool:
    """
    간단한 제목 판별 규칙
    """
    if not text:
        return False

    t = text.strip()

    # 너무 길면 제목이 아닐 가능성 높음
    if len(t) > 120:
        return False

    # markdown heading
    if t.startswith("#"):
        return True

    # 대문자 위주 제목
    if t.isupper() and len(t) >= 5:
        return True

    return False

# 5. PDF JSON에서 block 추출
- JSON 구조에서 페이지별 단락을 가져오는 함수

In [6]:
def extract_blocks_from_json(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    source_file = data.get("file", json_path.stem)

    blocks = []

    for page_info in data.get("pages", []):
        page_num = page_info.get("page")

        for para_idx, para in enumerate(page_info.get("paragraphs", []), start=1):
            text = clean_text(para)

            if not text:
                continue

            blocks.append({
                "source_file": source_file,
                "page": page_num,
                "block_id": f"{page_num}_{para_idx}",
                "raw_text": text
            })

    return blocks

# 6. 검색 품질을 높이는 chunk 생성 함수
- PDF 본문과 표를 분리해서 저장하는 핵심 chunk 함수
- 제목 + 본문 결합, 긴 블록 재분할, block_type 구분을 적용

In [7]:
def split_long_text(text: str, max_chars: int = 650):
    """
    긴 블록을 문장 기준으로 재분할
    """
    if len(text) <= max_chars:
        return [text]

    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent.strip()

    if current:
        chunks.append(current)

    return chunks


def build_pdf_documents_from_json_separated(json_path: Path):
    """
    JSON 1개를 읽어서 PDF 본문 Document / PDF 표 Document 리스트를 각각 반환
    """
    blocks = extract_blocks_from_json(json_path)

    text_docs = []
    table_docs = []
    current_heading = None

    for block in blocks:
        text = block["raw_text"]

        if is_heading(text):
            current_heading = text
            continue

        if is_noise_text(text):
            continue

        combined_text = text
        if current_heading:
            combined_text = f"{current_heading}\n\n{text}"

        block_type = "table" if "|" in text and "---" in text else "text"
        sub_chunks = split_long_text(combined_text, max_chars=650)

        for sub_idx, chunk in enumerate(sub_chunks, start=1):
            if is_noise_text(chunk):
                continue

            doc = Document(
                page_content=chunk,
                metadata={
                    "source_file": block["source_file"],
                    "page": block["page"],
                    "block_id": block["block_id"],
                    "sub_chunk_id": sub_idx,
                    "section_title": current_heading,
                    "block_type": block_type,
                    "data_type": "pdf_json"
                }
            )

            if block_type == "table":
                table_docs.append(doc)
            else:
                text_docs.append(doc)

    return text_docs, table_docs

# 7. 전체 PDF JSON 파일을 Document로 변환
- pages 수가 0인 JSON은 제외
- PDF 본문 Document와 PDF 표 Document를 각각 따로 수집

In [8]:
json_files = sorted(JSON_DIR.glob("*.json"))

pdf_text_docs = []
pdf_table_docs = []
skipped_json_files = []

print("JSON 파일 개수:", len(json_files))

for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    if len(data.get("pages", [])) == 0:
        skipped_json_files.append(jf.name)
        print(f"제외됨 (pages=0): {jf.name}")
        continue

    text_docs, table_docs = build_pdf_documents_from_json_separated(jf)

    pdf_text_docs.extend(text_docs)
    pdf_table_docs.extend(table_docs)

    print(f"{jf.name} → text={len(text_docs)} / table={len(table_docs)}")

print("\n전체 PDF 본문 Document 수:", len(pdf_text_docs))
print("전체 PDF 표 Document 수:", len(pdf_table_docs))
print("제외된 JSON 파일 수:", len(skipped_json_files))

JSON 파일 개수: 11
ADDP_Brochure_FINAL_122624_508c.json → text=36 / table=3
Costs_Fees.json → text=31 / table=14
Maternity_Br (1).json → text=30 / table=8
NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).json → text=90 / table=24
Overseas_HB(해외 프로그램 안내서).json → text=94 / table=18
Pharmacy_HB(tricare 약국 프로그램 안내서).json → text=219 / table=12
QLEs_FS (2).json → text=35 / table=3
Retiring_NGR_Br.json → text=36 / table=5
TFL_HB(평생 트라이케어).json → text=276 / table=17
TOP_Handbook_AUG_2023_FINAL_092223_508 (1).json → text=100 / table=9
TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).json → text=179 / table=8

전체 PDF 본문 Document 수: 1126
전체 PDF 표 Document 수: 121
제외된 JSON 파일 수: 0


# 8. PDF Document 샘플 확인
- PDF 본문 Document와 PDF 표 Document가 어떤 구조인지 각각 확인

In [9]:
if pdf_text_docs:
    sample = pdf_text_docs[0]

    print("[PDF 본문 샘플 metadata]")
    print(sample.metadata)

    print("\n[PDF 본문 샘플 text]")
    print(sample.page_content[:700])

if pdf_table_docs:
    sample_table = pdf_table_docs[0]

    print("\n[PDF 표 샘플 metadata]")
    print(sample_table.metadata)

    print("\n[PDF 표 샘플 text]")
    print(sample_table.page_content[:700])

[PDF 본문 샘플 metadata]
{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf', 'page': 1, 'block_id': '1_3', 'sub_chunk_id': 1, 'section_title': '# The care you need for dental health and readiness', 'block_type': 'text', 'data_type': 'pdf_json'}

[PDF 본문 샘플 text]
# The care you need for dental health and readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit for more information.

[PDF 표 샘플 metadata]
{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf', 'page': 1, 'block_id': '1_6', 'sub_chunk_id': 1, 'section_title': '# The care you need for dental health and readiness', 'block_type': 'table', 'data_type': 'pdf_json'}

[PDF 표 샘플 text]
# The care you need for dental health and readiness

| Online | < |
| --------- | -----------------------------------------------------------------------------------------

# 9. CSV 전처리 보조 함수
- row_to_text와 csv_type 분류 함수는 다음 셀에서 함께 정의

In [10]:
def row_to_text(row: pd.Series) -> str:
    parts = []

    for col, val in row.items():
        if pd.isna(val):
            continue

        val_str = str(val).strip()
        if not val_str:
            continue

        parts.append(f"{col}: {val_str}")

    return "\n".join(parts)

# 10. CSV를 Document로 변환
- CSV 유형을 coverage / exclusion / cost / plan / general로 구분
- CSV 파일 1개를 row 단위 Document 리스트로 변환

In [11]:
def get_csv_type(csv_path: Path) -> str:
    name = csv_path.name.lower()

    if "exclusion" in name:
        return "exclusion"
    elif "mental_health_services" in name:
        return "coverage"
    elif "health_plan_costs" in name:
        return "cost"
    elif "tricareplans" in name:
        return "plan"
    else:
        return "general"


def row_to_text(row: pd.Series) -> str:
    parts = []

    for col, val in row.items():
        if pd.isna(val):
            continue

        val_str = str(val).strip()
        if not val_str:
            continue

        parts.append(f"{col}: {val_str}")

    return "\n".join(parts)


def csv_file_to_documents(csv_path: Path) -> list[Document]:
    df = pd.read_csv(csv_path)
    docs = []
    csv_type = get_csv_type(csv_path)

    for idx, row in df.iterrows():
        text = clean_text(row_to_text(row))

        if is_noise_text(text):
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source_file": csv_path.name,
                    "row_index": int(idx),
                    "data_type": "csv",
                    "block_type": "row",
                    "csv_type": csv_type
                }
            )
        )

    return docs

# 11. 사용할 CSV 목록 지정
- 프로젝트에서 사용할 CSV 경로를 지정

In [12]:
csv_paths = [
    DATA_DIR / "mental_health_services.csv",
    DATA_DIR / "TricarePlans.csv",
    DATA_DIR / "tricare_exclusions.csv",
    DATA_DIR / "Health_Plan_Costs.csv"
]

for p in csv_paths:
    print(p.name, "존재:", p.exists())

mental_health_services.csv 존재: True
TricarePlans.csv 존재: True
tricare_exclusions.csv 존재: True
Health_Plan_Costs.csv 존재: True


# 12. 전체 CSV를 Document로 변환
- CSV를 coverage / exclusion / other로 분리해서 수집

In [13]:
csv_coverage_docs = []
csv_exclusion_docs = []
csv_other_docs = []

for csv_path in csv_paths:
    if not csv_path.exists():
        print("없음:", csv_path.name)
        continue

    docs = csv_file_to_documents(csv_path)
    csv_type = get_csv_type(csv_path)

    if csv_type == "coverage":
        csv_coverage_docs.extend(docs)
    elif csv_type == "exclusion":
        csv_exclusion_docs.extend(docs)
    else:
        csv_other_docs.extend(docs)

    print(f"{csv_path.name} ({csv_type}) → {len(docs)}개 Document")

print("\ncoverage docs:", len(csv_coverage_docs))
print("exclusion docs:", len(csv_exclusion_docs))
print("other docs:", len(csv_other_docs))

mental_health_services.csv (coverage) → 8개 Document
TricarePlans.csv (plan) → 10개 Document
tricare_exclusions.csv (exclusion) → 55개 Document
Health_Plan_Costs.csv (cost) → 234개 Document

coverage docs: 8
exclusion docs: 55
other docs: 244


# 13. CSV Document 샘플 확인
- coverage / exclusion / other 중 하나의 샘플 구조를 확인

In [14]:
if csv_coverage_docs:
    sample = csv_coverage_docs[0]
    print("[CSV coverage 샘플 metadata]")
    print(sample.metadata)
    print("\n[CSV coverage 샘플 text]")
    print(sample.page_content[:500])

if csv_exclusion_docs:
    sample_ex = csv_exclusion_docs[0]
    print("\n[CSV exclusion 샘플 metadata]")
    print(sample_ex.metadata)
    print("\n[CSV exclusion 샘플 text]")
    print(sample_ex.page_content[:500])

[CSV coverage 샘플 metadata]
{'source_file': 'mental_health_services.csv', 'row_index': 0, 'data_type': 'csv', 'block_type': 'row', 'csv_type': 'coverage'}

[CSV coverage 샘플 text]
서비스: Psychotherapy
내용: Psychotherapy is discussion-based mental health therapy. It’s also known as “therapy” and “counseling.” TRICARE covers office-based outpatient psychotherapy when medically or psychologically necessary to treat a mental health disorder. TRICARE covers inpatient and outpatient services, including: Individual sessions: Individual sessions are covered up to 60 minutes. Individual crisis sessions may extend up to 120 minutes. Family or conjoint sessions: Therapy designed to tre

[CSV exclusion 샘플 metadata]
{'source_file': 'tricare_exclusions.csv', 'row_index': 0, 'data_type': 'csv', 'block_type': 'row', 'csv_type': 'exclusion'}

[CSV exclusion 샘플 text]
name: Acupuncture
title: Acupuncture
content: TRICARE doesn’t coveracupuncture. This list of covered services isn’t all inclusive. TRICARE cove

# 14. 다국어 임베딩 모델 로드
- PDF/CSV 모두에 사용할 임베딩 모델을 로드

In [15]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("임베딩 모델 로드 완료")


임베딩 모델 로드 완료


# 15. PDF 전용 Chroma DB 생성
- PDF 본문 DB와 PDF 표 DB를 각각 따로 생성

In [16]:
vectorstore_pdf_text = Chroma.from_documents(
    documents=pdf_text_docs,
    embedding=embedding_model,
    persist_directory=str(PDF_TEXT_DB_DIR),
    collection_name=PDF_TEXT_COLLECTION_NAME
)

vectorstore_pdf_table = Chroma.from_documents(
    documents=pdf_table_docs,
    embedding=embedding_model,
    persist_directory=str(PDF_TABLE_DB_DIR),
    collection_name=PDF_TABLE_COLLECTION_NAME
)

print("PDF 본문 DB 생성 완료")
print("저장 경로:", PDF_TEXT_DB_DIR)
print("컬렉션 이름:", PDF_TEXT_COLLECTION_NAME)

print("\nPDF 표 DB 생성 완료")
print("저장 경로:", PDF_TABLE_DB_DIR)
print("컬렉션 이름:", PDF_TABLE_COLLECTION_NAME)

PDF 본문 DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_text_db
컬렉션 이름: tricare_pdf_json_text

PDF 표 DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_table_db
컬렉션 이름: tricare_pdf_json_table


# 16. CSV 전용 Chroma DB 생성
- CSV coverage / exclusion / other DB를 각각 따로 생성

In [17]:
vectorstore_csv_coverage = Chroma.from_documents(
    documents=csv_coverage_docs,
    embedding=embedding_model,
    persist_directory=str(CSV_COVERAGE_DB_DIR),
    collection_name=CSV_COVERAGE_COLLECTION_NAME
)

vectorstore_csv_exclusion = Chroma.from_documents(
    documents=csv_exclusion_docs,
    embedding=embedding_model,
    persist_directory=str(CSV_EXCLUSION_DB_DIR),
    collection_name=CSV_EXCLUSION_COLLECTION_NAME
)

vectorstore_csv_other = Chroma.from_documents(
    documents=csv_other_docs,
    embedding=embedding_model,
    persist_directory=str(CSV_OTHER_DB_DIR),
    collection_name=CSV_OTHER_COLLECTION_NAME
)

print("CSV coverage DB 생성 완료")
print("저장 경로:", CSV_COVERAGE_DB_DIR)
print("컬렉션 이름:", CSV_COVERAGE_COLLECTION_NAME)

print("\nCSV exclusion DB 생성 완료")
print("저장 경로:", CSV_EXCLUSION_DB_DIR)
print("컬렉션 이름:", CSV_EXCLUSION_COLLECTION_NAME)

print("\nCSV other DB 생성 완료")
print("저장 경로:", CSV_OTHER_DB_DIR)
print("컬렉션 이름:", CSV_OTHER_COLLECTION_NAME)

CSV coverage DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_coverage_db
컬렉션 이름: tricare_csv_coverage

CSV exclusion DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_exclusion_db
컬렉션 이름: tricare_csv_exclusion

CSV other DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_other_db
컬렉션 이름: tricare_csv_other


# 17. PDF DB 검색 테스트 (품질 확인)
- PDF 본문 DB를 우선 검색하고, 필요하면 PDF 표 DB도 별도로 검색

In [18]:
query = "Who is eligible for the Active Duty Dental Program?"

results = vectorstore_pdf_text.similarity_search(query, k=3)

for i, r in enumerate(results, start=1):
    print(f"\n=== PDF 본문 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])

table_results = vectorstore_pdf_table.similarity_search(query, k=2)

for i, r in enumerate(table_results, start=1):
    print(f"\n=== PDF 표 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])


=== PDF 본문 검색 결과 1 ===
metadata: {'page': 3, 'section_title': '# The Care You Need for Dental Readiness', 'block_type': 'text', 'source_file': 'TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf', 'data_type': 'pdf_json', 'block_id': '3_3', 'sub_chunk_id': 1}
text: # The Care You Need for Dental Readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. Its goal is to ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP for the Department of Defense.

=== PDF 본문 검색 결과 2 ===
metadata: {'page': 1, 'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf', 'block_id': '1_3', 'block_type': 'text', 'sub_chunk_id': 1, 'section_title': '# The care you need for dental health and readiness', 'data_type': 'pdf_json'}
text: # The care you need for dental health and readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental he

# 18. CSV DB 검색 테스트 (품질 확인)
- coverage 질문은 coverage DB를 우선 검색하고, exclusion DB도 별도로 확인

In [19]:
query = "mental health counseling coverage"

coverage_results = vectorstore_csv_coverage.similarity_search(query, k=3)

for i, r in enumerate(coverage_results, start=1):
    print(f"\n=== CSV coverage 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])

exclusion_results = vectorstore_csv_exclusion.similarity_search(query, k=2)

for i, r in enumerate(exclusion_results, start=1):
    print(f"\n=== CSV exclusion 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])

other_results = vectorstore_csv_other.similarity_search(query, k=2)

for i, r in enumerate(other_results, start=1):
    print(f"\n=== CSV other 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])


=== CSV coverage 검색 결과 1 ===
metadata: {'csv_type': 'coverage', 'row_index': 0, 'block_type': 'row', 'source_file': 'mental_health_services.csv', 'data_type': 'csv'}
text: 서비스: Psychotherapy
내용: Psychotherapy is discussion-based mental health therapy. It’s also known as “therapy” and “counseling.” TRICARE covers office-based outpatient psychotherapy when medically or psychologically necessary to treat a mental health disorder. TRICARE covers inpatient and outpatient services, including: Individual sessions: Individual sessions are covered up to 60 minutes. Individual crisis sessions may extend up to 120 minutes. Family or conjoint sessions: Therapy designed to tre

=== CSV coverage 검색 결과 2 ===
metadata: {'csv_type': 'coverage', 'row_index': 2, 'block_type': 'row', 'data_type': 'csv', 'source_file': 'mental_health_services.csv'}
text: 서비스: Psychological testing and assessment
내용: TRICARE covers psychological testing and assessment to help diagnose a patient and develop a treatment plan